# Fraud Detection for Microfinance — Model Notebook

This notebook trains a **fraud detection system** for microfinance transaction data.
Because the dataset has no pre-existing `IsFraud` label, we use a **hybrid approach**:

1. **Rule-based pseudo-labelling**: domain-knowledge heuristics flag highly suspicious transactions.
2. **Isolation Forest** (unsupervised): scores anomaly level for each transaction.
3. **Supervised classifier** (Random Forest): trained on pseudo-labels + anomaly features.
4. **Fraud risk regressor** (Gradient Boosting): outputs a 0–100 continuous risk score.

## Contents
1. Install dependencies
2. Load dataset
3. Explore & visualize
4. Feature engineering & pseudo-label creation
5. Preprocessing
6. Model 1 — Isolation Forest anomaly scorer (unsupervised baseline)
7. Model 2 — Random Forest fraud classifier (binary: fraud / not-fraud)
8. Model 3 — Gradient Boosting fraud risk score (continuous 0-100)
9. Save all artifacts to `fraud_detection_model/`
10. Test & validate saved models

## 1. Install dependencies

In [ ]:
!pip install -q pandas scikit-learn matplotlib seaborn joblib numpy

## 2. Load dataset

In [ ]:
import pandas as pd
from pathlib import Path

DATASETS_DIR = Path('../datasets')
MODELS_DIR = Path('../fraud_detection_model')
MODELS_DIR.mkdir(exist_ok=True)

csv_path = DATASETS_DIR / 'bank_transactions_data_2.csv'
df = pd.read_csv(csv_path)

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

## 3. Explore & visualize

In [ ]:
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nNumerical summary:')
df.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

sns.histplot(df['TransactionAmount'], bins=50, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Transaction Amount Distribution')

sns.histplot(df['LoginAttempts'], bins=10, ax=axes[0,1], color='coral')
axes[0,1].set_title('Login Attempts Distribution')

sns.countplot(x='TransactionType', data=df, ax=axes[0,2])
axes[0,2].set_title('Transaction Type')

sns.countplot(x='Channel', data=df, ax=axes[1,0])
axes[1,0].set_title('Channel')

sns.countplot(x='CustomerOccupation', data=df, ax=axes[1,1])
axes[1,1].set_title('Customer Occupation')

sns.histplot(df['TransactionDuration'], bins=50, ax=axes[1,2], color='green')
axes[1,2].set_title('Transaction Duration (secs)')

plt.tight_layout()
plt.show()

## 4. Feature engineering & pseudo-label creation

We engineer domain-driven features and create rule-based fraud pseudo-labels:

| Rule | Rationale |
|------|----------|
| `LoginAttempts >= 4` | Brute-force / account-takeover indicator |
| `TransactionAmount / AccountBalance > 0.8` | Draining the account |
| `TimeSinceLastTx < 0.03 days (~43 min)` | Velocity — rapid successive transactions |
| `TransactionAmount > mean + 3*std` | Unusually large individual transaction |
| `LoginAttempts >= 3 AND amount/balance > 0.5` | Combined suspicious signal |

A transaction is pseudo-labelled **Fraud = 1** if it triggers **any** rule.

In [ ]:
import numpy as np

# ── Parse date columns ────────────────────────────────────────────────────────
df['TransactionDate']         = pd.to_datetime(df['TransactionDate'])
df['PreviousTransactionDate'] = pd.to_datetime(df['PreviousTransactionDate'])

# ── Derived numeric features ──────────────────────────────────────────────────
# Days since previous transaction (velocity indicator)
df['DaysSinceLastTx'] = (
    df['TransactionDate'] - df['PreviousTransactionDate']
).dt.total_seconds() / 86400.0
# Clip to non-negative (data quality)
df['DaysSinceLastTx'] = df['DaysSinceLastTx'].abs().clip(lower=0)

# Ratio of transaction amount to account balance
df['AmountToBalanceRatio'] = df['TransactionAmount'] / (df['AccountBalance'] + 1e-6)

# Transaction hour (night-time transactions can be riskier)
df['TxHour'] = df['TransactionDate'].dt.hour

# Is night-time transaction (11pm – 5am)
df['IsNightTx'] = df['TxHour'].apply(lambda h: 1 if (h >= 23 or h <= 5) else 0)

# ── Encode categoricals ───────────────────────────────────────────────────────
tx_type_map = {'Credit': 0, 'Debit': 1}
channel_map = {'Online': 0, 'ATM': 1, 'Branch': 2}
occ_map = {v: i for i, v in enumerate(sorted(df['CustomerOccupation'].unique()))}

df['TxTypeCode']   = df['TransactionType'].map(tx_type_map).fillna(0).astype(int)
df['ChannelCode']  = df['Channel'].map(channel_map).fillna(0).astype(int)
df['OccupCode']    = df['CustomerOccupation'].map(occ_map).fillna(0).astype(int)

print('Occupation mapping:', occ_map)

# ── Rule-based pseudo-labels ──────────────────────────────────────────────────
amount_mean = df['TransactionAmount'].mean()
amount_std  = df['TransactionAmount'].std()

rule1 = df['LoginAttempts'] >= 4                                     # multiple failed logins
rule2 = df['AmountToBalanceRatio'] > 0.8                             # draining account
rule3 = df['DaysSinceLastTx'] < 0.03                                 # velocity (< ~43 min)
rule4 = df['TransactionAmount'] > (amount_mean + 3 * amount_std)     # outlier amount
rule5 = (df['LoginAttempts'] >= 3) & (df['AmountToBalanceRatio'] > 0.5)  # combined

df['IsFraud'] = (rule1 | rule2 | rule3 | rule4 | rule5).astype(int)

print('\nPseudo-label distribution:')
print(df['IsFraud'].value_counts())
print(f'Fraud rate: {df["IsFraud"].mean():.2%}')

# Rule contribution breakdown
print('\nIndividual rule triggers:')
for name, rule in [('LoginAttempts>=4', rule1), ('AmountToBalance>0.8', rule2),
                   ('Velocity<43min', rule3), ('OutlierAmount', rule4), ('Combined', rule5)]:
    print(f'  {name}: {rule.sum()} transactions')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['IsFraud'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','red'])
axes[0].set_title('Pseudo-labels: Fraud vs Legit')
axes[0].set_xticklabels(['Legit', 'Fraud'], rotation=0)

sns.boxplot(x='IsFraud', y='TransactionAmount', data=df, ax=axes[1])
axes[1].set_title('Amount by Fraud Label')
axes[1].set_xticklabels(['Legit', 'Fraud'])

sns.boxplot(x='IsFraud', y='LoginAttempts', data=df, ax=axes[2])
axes[2].set_title('Login Attempts by Fraud Label')
axes[2].set_xticklabels(['Legit', 'Fraud'])

plt.tight_layout()
plt.show()

## 5. Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

FEATURE_COLS = [
    'TransactionAmount',
    'TransactionDuration',
    'LoginAttempts',
    'AccountBalance',
    'CustomerAge',
    'DaysSinceLastTx',
    'AmountToBalanceRatio',
    'TxHour',
    'IsNightTx',
    'TxTypeCode',
    'ChannelCode',
    'OccupCode',
]

X = df[FEATURE_COLS].values.astype(np.float64)
y = df['IsFraud'].values

# Continuous target: fraud risk score 0-100 (blend anomaly signals)
# We scale rules: each rule contributes points, cap at 100
risk_score = (
    rule1.astype(float) * 30 +
    rule2.astype(float) * 25 +
    rule3.astype(float) * 20 +
    rule4.astype(float) * 15 +
    rule5.astype(float) * 10 +
    (df['LoginAttempts'] / 5.0) * 5 +
    df['AmountToBalanceRatio'].clip(0, 1) * 5
).clip(0, 100)
y_score = risk_score.values

# Feature names save
import joblib
joblib.dump(FEATURE_COLS, MODELS_DIR / 'fraud_feature_columns.pkl')

X_train, X_test, y_train, y_test, ys_train, ys_test = train_test_split(
    X, y, y_score, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

joblib.dump(scaler, MODELS_DIR / 'fraud_scaler.pkl')
joblib.dump({'tx_type': tx_type_map, 'channel': channel_map, 'occupation': occ_map},
            MODELS_DIR / 'fraud_encoders.pkl')

print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Train fraud rate:', y_train.mean().round(3))
print('Scaler + encoders saved.')

## 6. Model 1 — Isolation Forest anomaly scorer (unsupervised baseline)

**What it does**: Learns the normal transaction distribution without any labels.
Flags transactions far from the normal cluster as anomalies.

**Output**: `anomaly_score` — negative score, where more negative = more anomalous.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Contamination: expected fraction of fraud (from pseudo-labels)
contamination = float(y.mean())
contamination = max(0.01, min(contamination, 0.5))  # clip to valid range
print(f'Contamination: {contamination:.3f}')

iso = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_train_s)

# Predict: -1 = anomaly (fraud), 1 = normal
iso_pred = iso.predict(X_test_s)   # -1 or 1
iso_labels = (iso_pred == -1).astype(int)  # 1 = fraud

# Anomaly score (more negative = more anomalous); normalise to 0-1
iso_scores = iso.score_samples(X_test_s)     # negative
iso_scores_norm = (-iso_scores - (-iso_scores).min()) / ((-iso_scores).max() - (-iso_scores).min())

print('\n=== Isolation Forest vs pseudo-labels ===')
print(classification_report(y_test, iso_labels, target_names=['Legit', 'Fraud']))
try:
    print('ROC-AUC:', roc_auc_score(y_test, iso_scores_norm).round(4))
except Exception:
    pass

joblib.dump(iso, MODELS_DIR / 'fraud_isolation_forest.pkl')
print('Isolation Forest saved.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm = confusion_matrix(y_test, iso_labels)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[0], cmap='Blues',
            xticklabels=['Legit','Fraud'], yticklabels=['Legit','Fraud'])
axes[0].set_title('Isolation Forest Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Anomaly score distribution
sns.histplot(iso_scores_norm[y_test == 0], color='steelblue', label='Legit', bins=40, ax=axes[1], alpha=0.6)
sns.histplot(iso_scores_norm[y_test == 1], color='red', label='Fraud', bins=40, ax=axes[1], alpha=0.6)
axes[1].set_title('Anomaly Score Distribution')
axes[1].set_xlabel('Anomaly Score (0=normal, 1=anomalous)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Model 2 — Random Forest fraud classifier (binary)

**What it does**: Supervised binary classification (fraud / not-fraud) trained on pseudo-labels.
Uses class weighting to handle imbalance.

**Output**: `is_fraud` (bool) + `fraud_probability` (0.0–1.0)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_s, y_train)

y_pred = rf.predict(X_test_s)
y_prob = rf.predict_proba(X_test_s)[:, 1]

print('=== Random Forest Classifier ===')
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))
print('ROC-AUC:', roc_auc_score(y_test, y_prob).round(4))

joblib.dump(rf, MODELS_DIR / 'fraud_classifier.pkl')
print('Random Forest classifier saved.')

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Confusion matrix
cm2 = confusion_matrix(y_test, y_pred)
sns.heatmap(cm2, annot=True, fmt='d', ax=axes[0], cmap='Oranges',
            xticklabels=['Legit','Fraud'], yticklabels=['Legit','Fraud'])
axes[0].set_title('Random Forest Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Feature importances
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
importances.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Feature Importances (Random Forest)')

# ROC curve
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[2], name='RF Classifier')
axes[2].set_title('ROC Curve')

plt.tight_layout()
plt.show()

## 8. Model 3 — Gradient Boosting fraud risk score (0–100)

**What it does**: Regression that predicts a continuous fraud risk score (0 = no risk, 100 = definite fraud).
Built on the rule-weighted score so it reflects multiple combined signals.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

gbr = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
)
gbr.fit(X_train_s, ys_train)

ys_pred = gbr.predict(X_test_s).clip(0, 100)

print('=== Gradient Boosting Fraud Risk Scorer ===')
print(f'MAE:  {mean_absolute_error(ys_test, ys_pred):.2f}')
print(f'R^2:  {r2_score(ys_test, ys_pred):.4f}')

joblib.dump(gbr, MODELS_DIR / 'fraud_risk_scorer.pkl')
print('Gradient Boosting risk scorer saved.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Predicted vs actual risk score
axes[0].scatter(ys_test, ys_pred, alpha=0.3, s=10, color='purple')
axes[0].plot([0, 100], [0, 100], 'r--')
axes[0].set_xlabel('Actual Risk Score')
axes[0].set_ylabel('Predicted Risk Score')
axes[0].set_title('Risk Score: Predicted vs Actual')

# Distribution of predicted risk scores
sns.histplot(ys_pred, bins=50, ax=axes[1], color='purple', alpha=0.7)
axes[1].set_title('Distribution of Predicted Fraud Risk Scores')
axes[1].set_xlabel('Risk Score (0-100)')

plt.tight_layout()
plt.show()

## 9. Summary of saved artifacts

In [ ]:
import os
print('=== Artifacts in fraud_detection_model/ ===')
for f in sorted(MODELS_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:45s} {size_kb:8.1f} KB')

## 10. Test saved models (end-to-end validation)

In [ ]:
# Reload from disk and run inference on a synthetic suspicious transaction
iso_loaded  = joblib.load(MODELS_DIR / 'fraud_isolation_forest.pkl')
rf_loaded   = joblib.load(MODELS_DIR / 'fraud_classifier.pkl')
gbr_loaded  = joblib.load(MODELS_DIR / 'fraud_risk_scorer.pkl')
scaler_l    = joblib.load(MODELS_DIR / 'fraud_scaler.pkl')
encoders_l  = joblib.load(MODELS_DIR / 'fraud_encoders.pkl')
feat_cols_l = joblib.load(MODELS_DIR / 'fraud_feature_columns.pkl')

print('Feature columns:', feat_cols_l)

# Simulate a suspicious transaction (high login attempts, large amount vs balance)
suspicious = {
    'TransactionAmount': 4800.0,
    'TransactionDuration': 45,
    'LoginAttempts': 5,
    'AccountBalance': 5000.0,
    'CustomerAge': 28,
    'DaysSinceLastTx': 0.01,       # very recent = velocity
    'AmountToBalanceRatio': 0.96,
    'TxHour': 2,                   # 2am
    'IsNightTx': 1,
    'TxTypeCode': 1,               # Debit
    'ChannelCode': 0,              # Online
    'OccupCode': 3,                 # Student
}

# Normal transaction
normal = {
    'TransactionAmount': 45.0,
    'TransactionDuration': 120,
    'LoginAttempts': 1,
    'AccountBalance': 8000.0,
    'CustomerAge': 42,
    'DaysSinceLastTx': 3.5,
    'AmountToBalanceRatio': 0.006,
    'TxHour': 14,
    'IsNightTx': 0,
    'TxTypeCode': 1,
    'ChannelCode': 1,              # ATM
    'OccupCode': 1,                 # Engineer
}

def predict_transaction(tx_dict):
    vec = np.array([[tx_dict[c] for c in feat_cols_l]], dtype=np.float64)
    vec_s = scaler_l.transform(vec)
    
    iso_score = float(-iso_loaded.score_samples(vec_s)[0])
    rf_fraud  = bool(rf_loaded.predict(vec_s)[0])
    rf_prob   = float(rf_loaded.predict_proba(vec_s)[0][1])
    risk_score = float(gbr_loaded.predict(vec_s)[0].clip(0, 100))
    
    return {
        'is_fraud': rf_fraud,
        'fraud_probability': round(rf_prob, 4),
        'anomaly_score': round(iso_score, 4),
        'risk_score': round(risk_score, 1),
        'risk_level': 'HIGH' if risk_score >= 50 else ('MEDIUM' if risk_score >= 20 else 'LOW'),
    }

print('\n=== Suspicious transaction ===')
print(predict_transaction(suspicious))

print('\n=== Normal transaction ===')
print(predict_transaction(normal))